In [1]:
from lammps import lammps
import numpy as np

# Initialize LAMMPS
lmp = lammps()

# Define simulation settings (can be modified as needed)
lmp.command("units metal")
lmp.command("dimension 3")
lmp.command("boundary p p p")  # Periodic boundary conditions

# Create a substrate (e.g., simple fcc lattice for Cu)
lmp.command("lattice fcc 3.615")  # Lattice parameter for Cu
lmp.command("region box block 0 10 0 10 0 10")
lmp.command("create_box 1 box")

# Add Copper atoms to the simulation box
lmp.command("create_atoms 1 box")

# Set the potential (using the Embedded Atom Model for Cu)
lmp.command("pair_style eam")
lmp.command("pair_coeff * * Cu_u3.eam")  # Replace with the correct EAM potential file

# Set up temperature and dynamics
lmp.command("velocity all create 300.0 87287 loop geom")  # Set initial temperature
lmp.command("fix 1 all nve")  # Energy and volume conserving dynamics

# Bombard Cu atoms onto the substrate by adding them with velocity
lmp.command("group bombarders type 1")
lmp.command("velocity bombarders set 0.0 0.0 1.0 units box")  # Direction of bombardment along Z

# Time steps for the simulation
lmp.command("timestep 0.001")
lmp.command("run 10000")  # Number of steps for the simulation

# Collect data (e.g., energy, position, velocity)
lmp.command("thermo 100")
lmp.command("thermo_style custom step temp pe etotal")

# Extract and print the thermo data
for i in range(100):  # Adjust for number of steps you want to collect
    lmp.command("run 100")
    step = lmp.get_thermo("step")
    temp = lmp.get_thermo("temp")
    pe = lmp.get_thermo("pe")
    etotal = lmp.get_thermo("etotal")
    print(f"Step: {step}, Temp: {temp}, PE: {pe}, Etot: {etotal}")

# Close LAMMPS instance
lmp.close()


LAMMPS (29 Aug 2024 - Update 1)
Lattice spacing in x,y,z = 3.615 3.615 3.615
Created orthogonal box = (0 0 0) to (36.15 36.15 36.15)
  1 by 1 by 1 MPI processor grid
Created 4000 atoms
  using lattice units in orthogonal box = (0 0 0) to (36.15 36.15 36.15)
  create_atoms CPU = 0.002 seconds
ERROR on proc 0: cannot open eam potential file Cu_u3.eam: No such file or directory (src/src/potential_file_reader.cpp:58)
Last command: pair_coeff * * Cu_u3.eam


Exception: ERROR on proc 0: cannot open eam potential file Cu_u3.eam: No such file or directory (src/src/poten

## Idea Structure

### Bombarding
- Create a grid
- Randomly choose a cell of the grid and create a particle there
- The particle should have high energy because of the kinetic speed it had before it arrived

def CreateGrid
def CreateParticle
def energyOfCreatedParticle

### Motion on Surface
- Use Monte Carlo (MC) to get the correct material configuration

def MDSurface

### Surface
- Create the three layers with their own character traits

def FrozenLayer
def ThermostatLayer
def SurfaceLayer
def EvolveLayer

## Open Questions
- Kinetic Monte Carlo (MC)?
- How to bias the formation of a flat surface or an island?
    - Need to integrate physical values to take this into account, how most elegant?
- How to use LAMMPS?